# Monkey training (Kaggle)

Trains one Monkey model and, once training is finished, exports it for Ollama.

**Run it with _Save Version → Save & Run All (Commit)_** so it keeps going after you close the browser or shut down your laptop. Each run resumes from the last checkpoint on Hugging Face and stops safely before Kaggle's 12-hour limit. If it isn't finished, just run it again.

Settings (right-hand panel): **Accelerator: GPU T4 x2**, **Internet: on**, and under *Add-ons → Secrets* an `HF_TOKEN` secret (a Hugging Face **write** token) that's ticked for this notebook.

In [ ]:
# ---- settings ----
CONFIG = "configs/monkey-1.5b.json"      # or configs/monkey-0.5b.json
REPO_URL = "https://github.com/YOUR_GITHUB_NAME/MonkeyTrain.git"
BRANCH = "main"

In [ ]:
import os, subprocess, sys
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
url = REPO_URL
try:  # only needed if the GitHub repo is private
    gh = secrets.get_secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{gh}@")
except Exception:
    pass

def sh(cmd):
    shown = cmd
    for secret in (os.environ.get("HF_TOKEN"), gh if "gh" in globals() else None):
        if secret:
            shown = shown.replace(secret, "***")  # keep tokens out of the saved log
    print("$", shown, flush=True)
    subprocess.run(cmd, shell=True, check=True)

sh("rm -rf /kaggle/working/MonkeyTrain")
sh(f"git clone --depth 1 -b {BRANCH} {url} /kaggle/working/MonkeyTrain")
os.chdir("/kaggle/working/MonkeyTrain")
sh("pip install -q unsloth")
sh("pip install -q --no-deps -e .")

In [ ]:
# Train (or resume). Stops by itself before the time limit.
sh(f"{sys.executable} -m monkeytrain.train --config {CONFIG}")

In [ ]:
# Export to GGUF for Ollama, only once training has finished.
if os.path.exists("/kaggle/working/run/TRAINING_DONE"):
    sh(f"{sys.executable} -m monkeytrain.export --config {CONFIG}")
else:
    print("Training isn't finished yet. Run this notebook again (Save & Run All) to continue.")

In [ ]:
# Keep the notebook's saved output small (checkpoints live on Hugging Face).
sh("rm -rf /kaggle/working/run /kaggle/working/export /kaggle/working/MonkeyTrain")